# 07rating_refinement_comparison 노트북 목표

1. 06번에서 저장한 모델들을 test 리뷰에 적용해 이벤트 리뷰 여부를 예측한다.
2. 리뷰별 이벤트 예측 결과를 가게별 별점 평균 정제에 연결한다.
3. 중간 발표에서 채택한 후보 2 방식으로 별점을 정제한다.
   - 후보 2: 이벤트 확률이 높을수록 낮은 가중치로 평균 계산
4. 최종 해석은 프로젝트 제안 모델과 validation 기준 선택 모델 중심으로 진행한다.
5. 특정 가게 사례를 통해 후보 2의 정제 전후 별점이 실제로 어떻게 달라지는지 확인한다.

정제 후 별점의 정답 데이터가 없으므로, 후보 2 결과는 실제 서비스 적용 결과가 아니라 별점 정제 가능성을 확인하는 시뮬레이션으로 해석한다.


## 1. 라이브러리 로드

- 저장된 모델 bundle을 불러오고, test split 기준 예측과 별점 정제를 계산하기 위한 패키지를 임포트한다.
- 각 패키지의 역할은 다음과 같다.
  1. `joblib`: 05/06번에서 저장한 모델 bundle 로드
  2. `numpy`: 확률 기반 가중치와 배열 계산
  3. `pandas`: 리뷰별 예측표와 가게별 정제표 생성
  4. `display`: 노트북 표 출력
  5. `sklearn.metrics`: test 기준 분류 성능 계산
  6. `train_test_split`: 05/06번과 같은 test split 재현

In [1]:
import os

import joblib
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import average_precision_score, f1_score
from sklearn.model_selection import train_test_split


## 2. test 데이터와 원본 리뷰 데이터 연결

- `csv/reviews_embeddings_extract.csv`를 불러와 05/06번과 같은 방식으로 test split을 재현한다.
- 저장 모델 bundle의 `feature_cols`와 맞도록 raw KcBERT 임베딩과 메타데이터 컬럼을 그대로 사용한다.
- `store_name`이 비어 있으면 `store_url`을 가게 식별자로 사용한다.
- `rating`은 이벤트 예측 feature가 아니라, 가게별 기존 평균 별점과 정제 후 평균 별점을 계산할 때 사용한다.
- 현재 실행 데이터 기준의 `test 리뷰 수`, `test 가게 수`, `label 분포`는 아래 출력값으로 확인한다.
- `test split`: 모델 선택에는 사용하지 않고, 최종 참고 성능과 별점 정제 결과를 확인하는 데이터 구간이다.
- `store_id`: 가게별 평균 별점을 계산하기 위한 가게 식별자이다.


In [2]:
SEED = 42
LABEL_COL = 'label'
TEXT_COL = 'cleaned_review_text'
REVIEW_TEXT_COL = 'review_text'
STORE_COL = 'store_name'
STORE_URL_COL = 'store_url'
STORE_ID_COL = 'store_id'
RATING_COL = 'rating'

raw_df = pd.read_csv('csv/reviews_embeddings_extract.csv')

train_idx, temp_idx = train_test_split(
    raw_df.index,
    test_size=0.3,
    random_state=SEED,
    stratify=raw_df[LABEL_COL],
)
val_idx, test_idx = train_test_split(
    temp_idx,
    test_size=0.5,
    random_state=SEED,
    stratify=raw_df.loc[temp_idx, LABEL_COL],
)

y_test = raw_df.loc[test_idx, LABEL_COL].astype(int)

test_review_df = raw_df.loc[test_idx].copy().reset_index().rename(columns={'index': 'raw_index'})
test_review_df[RATING_COL] = pd.to_numeric(test_review_df[RATING_COL], errors='coerce')
test_review_df[STORE_ID_COL] = test_review_df[STORE_COL].fillna('').astype(str).str.strip()
test_review_df[STORE_ID_COL] = test_review_df[STORE_ID_COL].where(
    test_review_df[STORE_ID_COL].ne(''),
    test_review_df[STORE_URL_COL].fillna('').astype(str),
)
test_review_df[STORE_COL] = test_review_df[STORE_COL].fillna('').astype(str).str.strip()
test_review_df[STORE_COL] = test_review_df[STORE_COL].where(
    test_review_df[STORE_COL].ne(''),
    test_review_df[STORE_ID_COL],
)

text_test = test_review_df[TEXT_COL].fillna('').astype(str)

test_review_count = len(test_review_df)
test_store_count = test_review_df[STORE_ID_COL].nunique()
test_label_distribution = y_test.value_counts().sort_index().to_dict()

print('test 리뷰 수:', test_review_count)
print('test 가게 수:', test_store_count)
print('test label 분포:', test_label_distribution)


test 리뷰 수: 1327
test 가게 수: 118
test label 분포: {0: 854, 1: 473}


C:\Users\ICT\AppData\Local\Temp\ipykernel_23224\2698127778.py:10: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  raw_df = pd.read_csv('csv/reviews_embeddings_extract.csv')


## 3. 비교 모델 로드

07번은 별점 정제 단계이므로 제안 모델과 선택 모델만 같은 test 리뷰에 적용한다.

1. `final_proposed_model`: 프로젝트 제안 모델, KcBERT PCA + 메타데이터 hybrid
2. `final_selected_model`: 06번에서 validation F1 기준으로 선택된 모델

`final_proposed_model`과 `final_selected_model`은 06번에서 저장한 bundle을 사용하므로, 실제 모델명과 feature set은 bundle에서 동적으로 읽어온다.


In [3]:
model_specs = [
    {
        'model_key': 'final_proposed_model',
        'model_label': '제안 모델',
        'model_role': 'proposed',
        'path': 'outputs/final_proposed_model.joblib',
        'input_type': 'tabular',
    },
    {
        'model_key': 'final_selected_model',
        'model_label': '선택 모델',
        'model_role': 'selected',
        'path': 'outputs/final_selected_model.joblib',
        'input_type': 'tabular',
    },
]

missing_paths = [spec['path'] for spec in model_specs if not os.path.exists(spec['path'])]
if missing_paths:
    raise FileNotFoundError(f'필요한 모델 파일이 없습니다: {missing_paths}')

model_bundles = {spec['model_key']: joblib.load(spec['path']) for spec in model_specs}

proposed_bundle = model_bundles['final_proposed_model']
proposed_name = proposed_bundle.get('model_name', 'final_proposed_model')
proposed_feature_set = proposed_bundle.get('feature_set')
proposed_label = f"제안 모델 ({proposed_name}: {proposed_feature_set})" if proposed_feature_set else f"제안 모델 ({proposed_name})"

selected_bundle = model_bundles['final_selected_model']
selected_name = selected_bundle.get('model_name', 'final_selected_model')
selected_feature_set = selected_bundle.get('feature_set')
selected_label = f"선택 모델 ({selected_name}: {selected_feature_set})" if selected_feature_set else f"선택 모델 ({selected_name})"
for spec in model_specs:
    if spec['model_key'] == 'final_proposed_model':
        spec['model_label'] = proposed_label
        spec['input_type'] = proposed_bundle.get('input_type', spec['input_type'])
    if spec['model_key'] == 'final_selected_model':
        spec['model_label'] = selected_label
        spec['input_type'] = selected_bundle.get('input_type', spec['input_type'])

loaded_models = pd.DataFrame([
    {
        'model_key': spec['model_key'],
        'model_label': spec['model_label'],
        'model_role': spec['model_role'],
        'input_type': spec['input_type'],
        'path': spec['path'],
    }
    for spec in model_specs
])
display(loaded_models)


,model_key,model_label,model_role,input_type,path
0,final_proposed_model,제안 모델 (proposed_hybrid_mlp_04: kcbert_pca+meta...,proposed,tabular,outputs/final_proposed_model.joblib
1,final_selected_model,선택 모델 (baseline_2_metadata_only_mlp: metadata_...,selected,tabular,outputs/final_selected_model.joblib


## 4. test 리뷰별 이벤트 예측 계산

- 각 모델에 test 리뷰를 넣어 이벤트 리뷰일 확률을 계산한다.
- 저장된 threshold 이상이면 이벤트 리뷰로 예측한다.
- 이 섹션에서는 test 리뷰가 총 몇 개인지, 실제 이벤트 리뷰가 몇 개인지, 각 모델이 몇 개를 이벤트 리뷰로 예측했는지 확인한다.
- 실제 이벤트 리뷰 중 모델이 몇 개를 잡았는지 `이벤트 탐지율(Recall)`로 확인한다.
- 이벤트라고 예측한 리뷰 중 실제 이벤트 리뷰가 몇 개인지 `이벤트 예측 정확도(Precision)`로 확인한다.
- 별점 정제는 여기서 만든 리뷰별 `event_prob`를 기반으로 진행한다.


In [4]:
def predict_event_probability(spec, bundle):
    model = bundle['model']

    if spec['input_type'] == 'text':
        return model.predict_proba(text_test)[:, 1]

    feature_cols = bundle.get('feature_cols')
    if feature_cols is None:
        raise ValueError(f"{spec['model_label']} bundle에 feature_cols가 없습니다.")

    missing_cols = [col for col in feature_cols if col not in test_review_df.columns]
    if missing_cols:
        raise KeyError(f"{spec['model_label']} 입력 feature가 test raw 데이터에 없습니다: {missing_cols[:10]}")

    return model.predict_proba(test_review_df[feature_cols])[:, 1]


def make_error_type(actual, pred):
    return np.select(
        [
            (actual == 1) & (pred == 1),
            (actual == 0) & (pred == 0),
            (actual == 0) & (pred == 1),
            (actual == 1) & (pred == 0),
        ],
        ['TP', 'TN', 'FP', 'FN'],
        default='UNKNOWN',
    )


prediction_frames = []
summary_rows = []
actual_label = y_test.to_numpy()

actual_event_count = int((actual_label == 1).sum())
actual_normal_count = int((actual_label == 0).sum())

actual_label_summary = pd.DataFrame([{
    'test 리뷰 수': len(test_review_df),
    '실제 일반 리뷰 수': actual_normal_count,
    '실제 이벤트 리뷰 수': actual_event_count,
    '실제 이벤트 비율': float(actual_event_count / len(test_review_df)),
}]).round({'실제 이벤트 비율': 4})

for spec in model_specs:
    bundle = model_bundles[spec['model_key']]
    threshold = float(bundle.get('best_threshold', bundle.get('default_threshold', 0.5)))

    event_prob = predict_event_probability(spec, bundle)
    event_pred = (event_prob >= threshold).astype(int)
    candidate_2_weight = np.clip(1 - event_prob, 0, 1)

    event_pred_count = int(event_pred.sum())
    captured_event_count = int(((actual_label == 1) & (event_pred == 1)).sum())
    false_event_pred_count = int(((actual_label == 0) & (event_pred == 1)).sum())
    event_recall = captured_event_count / actual_event_count if actual_event_count > 0 else 0.0
    event_precision = captured_event_count / event_pred_count if event_pred_count > 0 else 0.0

    prediction_frames.append(pd.DataFrame({
        'raw_index': test_review_df['raw_index'].to_numpy(),
        'store_id': test_review_df[STORE_ID_COL].to_numpy(),
        'store_name': test_review_df[STORE_COL].to_numpy(),
        'store_url': test_review_df[STORE_URL_COL].to_numpy(),
        'rating': test_review_df[RATING_COL].to_numpy(),
        'review_text': test_review_df[REVIEW_TEXT_COL].fillna('').astype(str).to_numpy(),
        'cleaned_review_text': test_review_df[TEXT_COL].fillna('').astype(str).to_numpy(),
        'actual_label': actual_label,
        'actual_label_name': np.where(actual_label == 1, '이벤트 리뷰(1)', '일반 리뷰(0)'),
        'model_key': spec['model_key'],
        'model_label': spec['model_label'],
        'model_role': spec['model_role'],
        'threshold': threshold,
        'event_prob': event_prob,
        'event_pred': event_pred,
        'event_pred_name': np.where(event_pred == 1, '이벤트 예측(1)', '일반 예측(0)'),
        'is_correct': actual_label == event_pred,
        'error_type': make_error_type(actual_label, event_pred),
        'candidate_2_weight': candidate_2_weight,
        'candidate_2_weighted_rating': test_review_df[RATING_COL].to_numpy() * candidate_2_weight,
    }))

    summary_rows.append({
        'model_label': spec['model_label'],
        'model_role': spec['model_role'],
        'threshold': threshold,
        'actual_event_count': actual_event_count,
        'event_pred_count': event_pred_count,
        'captured_event_count': captured_event_count,
        'false_event_pred_count': false_event_pred_count,
        'event_pred_rate': float(event_pred.mean()),
        'event_recall': event_recall,
        'event_precision': event_precision,
        'f1': float(f1_score(actual_label, event_pred)),
        'pr_auc': float(average_precision_score(actual_label, event_prob)),
    })


prediction_detail = pd.concat(prediction_frames, ignore_index=True)
prediction_summary = pd.DataFrame(summary_rows).round({
    'threshold': 4,
    'event_pred_rate': 4,
    'event_recall': 4,
    'event_precision': 4,
    'f1': 4,
    'pr_auc': 4,
})

expected_rows = len(test_review_df) * len(model_specs)
assert len(prediction_detail) == expected_rows
np.testing.assert_allclose(
    prediction_detail['candidate_2_weight'].to_numpy(),
    1 - prediction_detail['event_prob'].to_numpy(),
)

prediction_summary_display = prediction_summary.rename(columns={
    'model_label': '모델',
    'model_role': '모델 구분',
    'threshold': '임계값',
    'actual_event_count': '실제 이벤트 수',
    'event_pred_count': '이벤트 예측 수',
    'captured_event_count': '실제 이벤트 중 탐지 수',
    'false_event_pred_count': '일반을 이벤트로 예측 수',
    'event_pred_rate': '이벤트 예측 비율',
    'event_recall': '이벤트 탐지율(Recall)',
    'event_precision': '이벤트 예측 정확도(Precision)',
    'f1': 'F1',
    'pr_auc': 'PR-AUC',
})

print('test 실제 라벨 분포')
display(actual_label_summary)

print('모델별 이벤트 탐지 요약')
display(prediction_summary_display)


test 실제 라벨 분포


,test 리뷰 수,실제 일반 리뷰 수,실제 이벤트 리뷰 수,실제 이벤트 비율
0,1327,854,473,0.3564


모델별 이벤트 탐지 요약


,모델,모델 구분,임계값,실제 이벤트 수,이벤트 예측 수,실제 이벤트 중 탐지 수,일반을 이벤트로 예측 수,이벤트 예측 비율,이벤트 탐지율(Recall),이벤트 예측 정확도(Precision),F1,PR-AUC
0,제안 모델 (proposed_hybrid_mlp_04: kcbert_pca+meta...,proposed,0.5009,473,586,233,353,0.4416,0.4926,0.3976,0.4400,0.4029
1,선택 모델 (baseline_2_metadata_only_mlp: metadata_...,selected,0.5011,473,983,365,618,0.7408,0.7717,0.3713,0.5014,0.3564


- `event_prob`: 해당 리뷰가 이벤트 리뷰일 확률이다.
- `event_pred`: threshold 기준으로 이벤트 리뷰라고 판단했는지 여부이다.
- `threshold`: 이벤트 리뷰로 판단하는 확률 기준이다.
- `F1`: 이벤트 리뷰 예측의 precision과 recall을 함께 반영한 균형 지표이다. 이벤트라고 예측한 것이 얼마나 맞았는지와 실제 이벤트를 얼마나 놓치지 않았는지를 동시에 본다.
- `PR-AUC`: 여러 threshold에서 precision과 recall의 관계를 종합한 지표이다. 이벤트 리뷰처럼 관심 클래스 탐지가 중요한 문제에서 보조 지표로 사용한다.


## 5. 리뷰 단위 예측 예시

- test 리뷰 중 일부를 뽑아 제안 모델과 선택 모델의 이벤트 예측 결과를 나란히 확인한다.
- 각 리뷰에 대해 실제 라벨, 모델별 예측값, 이벤트 확률, 정답 여부를 비교한다.
- `EXAMPLE_REVIEW_COUNT`는 최대 샘플 수이며, 실제 출력 개수는 현재 test 리뷰 수와 비교해 자동으로 정한다.
- 전체 상세 결과는 `main_review_predictions_detail`에 저장하고, 화면에는 확인용 요약 표만 출력한다.


In [5]:
review_detail_cols = [
    'model_label',
    'raw_index',
    'store_id',
    'store_name',
    'rating',
    'review_text',
    'cleaned_review_text',
    'actual_label_name',
    'event_pred_name',
    'event_prob',
    'threshold',
    'is_correct',
    'error_type',
    'candidate_2_weight',
    'candidate_2_weighted_rating',
]

main_review_predictions = (
    prediction_detail[prediction_detail['model_role'].isin(['proposed', 'selected'])]
    .sort_values(['raw_index', 'model_role'])
    .reset_index(drop=True)
)

main_review_predictions_detail = main_review_predictions[review_detail_cols].round({
    'event_prob': 4,
    'threshold': 4,
    'candidate_2_weight': 4,
    'candidate_2_weighted_rating': 4,
})

EXAMPLE_REVIEW_COUNT = 5
EXAMPLE_REVIEW_SEED = 42
actual_example_count = min(EXAMPLE_REVIEW_COUNT, test_review_df['raw_index'].nunique())
example_raw_indices = (
    test_review_df['raw_index']
    .drop_duplicates()
    .sample(n=actual_example_count, random_state=EXAMPLE_REVIEW_SEED)
)

main_review_predictions_example = main_review_predictions[
    main_review_predictions['raw_index'].isin(example_raw_indices)
].copy()

example_base = (
    main_review_predictions_example
    .drop_duplicates('raw_index')
    [['raw_index', 'rating', 'review_text', 'actual_label_name']]
    .rename(columns={
        'raw_index': '리뷰번호',
        'rating': '별점',
        'review_text': '리뷰',
        'actual_label_name': '실제 라벨',
    })
)

proposed_example = (
    main_review_predictions_example[main_review_predictions_example['model_role'] == 'proposed']
    [['raw_index', 'event_pred_name', 'event_prob', 'is_correct']]
    .rename(columns={
        'raw_index': '리뷰번호',
        'event_pred_name': '제안 모델 예측',
        'event_prob': '제안 모델 이벤트성 확률',
        'is_correct': '제안 모델 정답 여부',
    })
)

selected_example = (
    main_review_predictions_example[main_review_predictions_example['model_role'] == 'selected']
    [['raw_index', 'event_pred_name', 'event_prob', 'is_correct']]
    .rename(columns={
        'raw_index': '리뷰번호',
        'event_pred_name': '선택 모델 예측',
        'event_prob': '선택 모델 이벤트성 확률',
        'is_correct': '선택 모델 정답 여부',
    })
)

main_review_predictions_simple = (
    example_base
    .merge(proposed_example, on='리뷰번호', how='left')
    .merge(selected_example, on='리뷰번호', how='left')
    .sort_values('리뷰번호')
    .round({
        '제안 모델 이벤트성 확률': 4,
        '선택 모델 이벤트성 확률': 4,
    })
)

print(f'랜덤 리뷰 예측 예시 수: {actual_example_count}개 / test 리뷰 수: {len(test_review_df)}개')
display(main_review_predictions_simple)


랜덤 리뷰 예측 예시 수: 5개 / test 리뷰 수: 1327개


,리뷰번호,별점,리뷰,실제 라벨,제안 모델 예측,제안 모델 이벤트성 확률,제안 모델 정답 여부,선택 모델 예측,선택 모델 이벤트성 확률,선택 모델 정답 여부
0,873,5.0,양 많아요! 맛있게 잘 먹었습니다!,이벤트 리뷰(1),일반 예측(0),0.4744,False,일반 예측(0),0.4394,False
1,1115,5.0,맛있게 잘 먹었습니다!\n감사합니다!,일반 리뷰(0),일반 예측(0),0.4303,True,이벤트 예측(1),0.5079,False
2,4457,5.0,2월에만 3번 주문했어요 ㅎㅎ\n커피 너무 꼬소하고 캔에 넣어주셔서 좋습니다.\n여...,일반 리뷰(0),이벤트 예측(1),0.5202,False,이벤트 예측(1),0.5107,False
3,5032,5.0,파스타 후기가 너무 좋아서 시켰는데 아니나 다를까 너무 탁월한 선택이었어요!!!! ...,일반 리뷰(0),일반 예측(0),0.4347,True,이벤트 예측(1),0.5145,False
4,7904,5.0,너무 맛있어용,일반 리뷰(0),이벤트 예측(1),0.5816,False,일반 예측(0),0.4378,True


## 6. 후보 2 별점 정제 함수 정의

가게별 기존 평균 별점을 기준으로 후보 2 정제 방식을 계산한다.

- 후보 2는 `1 - event_prob`를 가중치로 사용한다.
- 이벤트 확률이 높을수록 해당 리뷰의 별점 반영 비중이 작아진다.
- 특정 가게의 가중치 합이 0이면 기존 평균 별점을 유지한다.


In [6]:
def original_store_rating(frame):
    return (
        frame.groupby('store_id')
        .agg(
            store_name=('store_name', 'first'),
            review_count=('rating', 'size'),
            original_mean_rating=('rating', 'mean'),
        )
        .reset_index()
    )


def candidate_2_probability_weighted(frame):
    work = frame.copy()
    work['weight'] = work['candidate_2_weight']
    work['weighted_rating'] = work['candidate_2_weighted_rating']

    original = original_store_rating(work)
    weighted = (
        work.groupby('store_id')
        .agg(
            weight_sum=('weight', 'sum'),
            weighted_rating_sum=('weighted_rating', 'sum'),
        )
        .reset_index()
    )
    result = original.merge(weighted, on='store_id', how='left')
    result['fallback_used'] = result['weight_sum'].fillna(0) <= 0
    result['refined_rating'] = np.where(
        result['fallback_used'],
        result['original_mean_rating'],
        result['weighted_rating_sum'] / result['weight_sum'],
    )
    return result


def summarize_refinement(store_result, frame, model_info, candidate_key, candidate_label):
    merged = store_result.copy()
    merged['rating_delta'] = merged['refined_rating'] - merged['original_mean_rating']
    metrics = prediction_summary[prediction_summary['model_label'] == model_info['model_label']].iloc[0]
    return {
        'model_key': model_info['model_key'],
        'model_label': model_info['model_label'],
        'model_role': model_info['model_role'],
        'candidate': candidate_key,
        'candidate_label': candidate_label,
        'threshold': float(frame['threshold'].iloc[0]),
        'store_count': int(merged['store_id'].nunique()),
        'review_count': int(len(frame)),
        'f1': float(metrics['f1']),
        'pr_auc': float(metrics['pr_auc']),
        'event_pred_count': int(frame['event_pred'].sum()),
        'event_pred_rate': float(frame['event_pred'].mean()),
        'original_mean_rating': float(frame['rating'].mean()),
        'refined_mean_rating': float(merged['refined_rating'].mean()),
        'mean_rating_delta': float(merged['rating_delta'].mean()),
        'mean_abs_rating_delta': float(merged['rating_delta'].abs().mean()),
        'max_abs_rating_delta': float(merged['rating_delta'].abs().max()),
        'fallback_store_count': int(merged['fallback_used'].sum()),
    }


## 7. 제안 모델과 선택 모델에 후보 2 적용

- 제안 모델과 선택 모델 각각에 후보 2 확률 기반 가중 평균을 적용한다.
- 이 단계에서는 이후 표에서 사용할 `all_store_results`와 `refinement_summary`만 만든다.
- 실제 표 출력은 10번 메인 비교표와 11번 저장용 비교표에서 진행한다.


In [7]:
store_results = []
summary_rows = []

CANDIDATE_KEY = 'candidate_2_probability_weighted'
CANDIDATE_LABEL = '후보 2: 이벤트 확률 기반 가중 평균'

for spec in model_specs:
    frame = prediction_detail[prediction_detail['model_key'] == spec['model_key']].copy()

    store_result = candidate_2_probability_weighted(frame)
    store_result['model_key'] = spec['model_key']
    store_result['model_label'] = spec['model_label']
    store_result['model_role'] = spec['model_role']
    store_result['candidate'] = CANDIDATE_KEY
    store_result['candidate_label'] = CANDIDATE_LABEL
    store_result['rating_delta'] = store_result['refined_rating'] - store_result['original_mean_rating']

    store_results.append(store_result)
    summary_rows.append(summarize_refinement(
        store_result,
        frame,
        spec,
        CANDIDATE_KEY,
        CANDIDATE_LABEL,
    ))

all_store_results = pd.concat(store_results, ignore_index=True)
refinement_summary = pd.DataFrame(summary_rows)

assert len(refinement_summary) == len(model_specs), '모델별 후보 2 결과가 1개씩 있어야 한다.'

print('후보 2 정제 결과 생성 완료:', refinement_summary.shape)


후보 2 정제 결과 생성 완료: (2, 18)


## 8. 리뷰 예측과 가게별 후보 2 정제 결과 연결

- 리뷰별 이벤트 예측 결과와 해당 리뷰가 속한 가게의 후보 2 정제 후 별점을 연결한다.
- 한 리뷰의 이벤트 확률이 가중치로 반영된 뒤, 그 가게의 평균 별점이 어떻게 바뀌었는지 확인할 수 있다.
- 이후 특정 가게 예시와 오답 샘플 확인에서 이 연결 테이블을 사용한다.


In [8]:
candidate_2_store = (
    all_store_results[all_store_results['candidate'] == 'candidate_2_probability_weighted']
    [[
        'model_key',
        'store_id',
        'original_mean_rating',
        'refined_rating',
        'rating_delta',
    ]]
    .rename(columns={
        'original_mean_rating': 'original_store_rating',
        'refined_rating': 'candidate_2_store_refined_rating',
        'rating_delta': 'candidate_2_store_rating_delta',
    })
)

review_refinement_detail = prediction_detail.merge(candidate_2_store, on=['model_key', 'store_id'], how='left')

assert len(review_refinement_detail) == len(prediction_detail)

review_refinement_cols = [
    'model_label',
    'raw_index',
    'store_id',
    'store_name',
    'rating',
    'review_text',
    'cleaned_review_text',
    'actual_label_name',
    'event_pred_name',
    'event_prob',
    'is_correct',
    'error_type',
    'candidate_2_weight',
    'original_store_rating',
    'candidate_2_store_refined_rating',
    'candidate_2_store_rating_delta',
]

simple_refinement_cols = [
    'model_label',
    'rating',
    'review_text',
    'actual_label_name',
    'event_pred_name',
    'event_prob',
    'candidate_2_weight',
    'original_store_rating',
    'candidate_2_store_refined_rating',
]

simple_refinement_column_names = {
    'model_label': '모델',
    'rating': '리뷰별점',
    'review_text': '리뷰',
    'actual_label_name': '실제',
    'event_pred_name': '예측',
    'event_prob': '이벤트확률',
    'candidate_2_weight': '후보2가중치',
    'original_store_rating': '기존가게별점',
    'candidate_2_store_refined_rating': '후보2정제별점',
}

main_review_refinement_detail = (
    review_refinement_detail[review_refinement_detail['model_role'].isin(['proposed', 'selected'])]
    .sort_values(['raw_index', 'model_role'])
    .reset_index(drop=True)
)

main_review_refinement_full = main_review_refinement_detail[review_refinement_cols].round({
    'event_prob': 4,
    'candidate_2_weight': 4,
    'original_store_rating': 4,
    'candidate_2_store_refined_rating': 4,
    'candidate_2_store_rating_delta': 4,
})

main_review_refinement_simple = (
    main_review_refinement_detail[simple_refinement_cols]
    .rename(columns=simple_refinement_column_names)
    .round({
        '이벤트확률': 4,
        '후보2가중치': 4,
        '기존가게별점': 4,
        '후보2정제별점': 4,
    })
)

print('리뷰별 예측과 후보 2 정제 별점 연결 테이블 생성 완료:', main_review_refinement_simple.shape)


리뷰별 예측과 후보 2 정제 별점 연결 테이블 생성 완료: (2654, 9)


## 9. 특정 가게 후보 2 정제 결과 확인

- test 데이터에서 후보 2 적용 후 별점 변화량이 큰 가게를 하나 선택해 정제 전후 별점을 확인한다.
- `STORE_EXAMPLE_ID`를 지정하면 특정 가게를 직접 볼 수 있고, 비워두면 별점 변화량이 큰 가게를 자동 선택한다.
- 별점 변화량이 같은 경우에는 리뷰 수가 많은 가게를 우선 선택한다.


In [9]:
STORE_EXAMPLE_ID = None
STORE_EXAMPLE_MODEL_ROLES = ['proposed', 'selected']

store_example_pool = (
    all_store_results[all_store_results['model_role'].isin(STORE_EXAMPLE_MODEL_ROLES)]
    .groupby('store_id')
    .agg(
        store_name=('store_name', 'first'),
        review_count=('review_count', 'max'),
        max_abs_rating_delta=('rating_delta', lambda s: s.abs().max()),
    )
    .reset_index()
    .sort_values(['max_abs_rating_delta', 'review_count'], ascending=[False, False])
    .reset_index(drop=True)
)

if STORE_EXAMPLE_ID is None:
    selected_store_id = store_example_pool.iloc[0]['store_id']
else:
    selected_store_id = STORE_EXAMPLE_ID

store_example_detail = review_refinement_detail[
    review_refinement_detail['model_role'].isin(STORE_EXAMPLE_MODEL_ROLES)
    & review_refinement_detail['store_id'].eq(selected_store_id)
].copy()

if store_example_detail.empty:
    raise ValueError(f'선택한 가게가 test 데이터에 없습니다: {selected_store_id}')

store_example_rows = []
for (model_key, model_label), group in store_example_detail.groupby(['model_key', 'model_label']):
    original_mean = group['rating'].mean()
    weight_sum = group['candidate_2_weight'].sum()
    refined_rating = (
        original_mean
        if weight_sum <= 0
        else (group['rating'] * group['candidate_2_weight']).sum() / weight_sum
    )
    rating_delta = refined_rating - original_mean
    store_example_rows.append({
        '모델': model_label,
        '리뷰 수': len(group),
        '이벤트 예측 수': int(group['event_pred'].sum()),
        '기존 평균 별점': original_mean,
        '후보 2 정제 후 별점': refined_rating,
        '별점 변화량': rating_delta,
        '변화 요약': f'{original_mean:.4f} -> {refined_rating:.4f} ({rating_delta:+.4f})',
    })

store_rating_refinement_example = pd.DataFrame(store_example_rows).round({
    '기존 평균 별점': 4,
    '후보 2 정제 후 별점': 4,
    '별점 변화량': 4,
})

store_name_for_print = store_example_detail['store_name'].iloc[0]
print('선택 가게:', store_name_for_print)
print('store_id:', selected_store_id)
print('선택 기준: 별점 변화량이 큰 가게 우선, 동률이면 리뷰 수가 많은 가게')
display(store_rating_refinement_example)


선택 가게: https://s.baemin.com/pG000flG524ai
store_id: https://s.baemin.com/pG000flG524ai
선택 기준: 별점 변화량이 큰 가게 우선, 동률이면 리뷰 수가 많은 가게


,모델,리뷰 수,이벤트 예측 수,기존 평균 별점,후보 2 정제 후 별점,별점 변화량,변화 요약
0,제안 모델 (proposed_hybrid_mlp_04: kcbert_pca+meta...,4,1,3.75,3.3334,-0.4166,3.7500 -> 3.3334 (-0.4166)
1,선택 모델 (baseline_2_metadata_only_mlp: metadata_...,4,3,3.75,3.8345,0.0845,3.7500 -> 3.8345 (+0.0845)


## 10. 메인 비교표: 제안 모델 vs 선택 모델

- 제안 모델과 validation 기준 선택 모델의 후보 2 정제 결과를 비교한다.
- 이벤트 예측 수, 기존 평균 별점, 정제 후 평균 별점, 별점 변화량을 확인한다.
- 이 표를 07번의 최종 해석 기준으로 사용한다.

In [10]:
main_roles = ['proposed', 'selected']
main_comparison = (
    refinement_summary[refinement_summary['model_role'].isin(main_roles)]
    .sort_values(['model_role', 'candidate'])
    .reset_index(drop=True)
)

main_comparison_display = (
    main_comparison[[
        'model_label',
        'model_role',
        'threshold',
        'f1',
        'pr_auc',
        'event_pred_count',
        'event_pred_rate',
        'original_mean_rating',
        'refined_mean_rating',
        'mean_rating_delta',
        'mean_abs_rating_delta',
        'max_abs_rating_delta',
    ]]
    .rename(columns={
        'model_label': '모델',
        'model_role': '모델 구분',
        'threshold': '임계값',
        'f1': 'F1',
        'pr_auc': 'PR-AUC',
        'event_pred_count': '이벤트 예측 수',
        'event_pred_rate': '이벤트 예측 비율',
        'original_mean_rating': '기존 평균 별점',
        'refined_mean_rating': '정제 후 평균 별점',
        'mean_rating_delta': '평균 별점 변화량',
        'mean_abs_rating_delta': '평균 절대 변화량',
        'max_abs_rating_delta': '최대 절대 변화량',
    })
    .round({
        '임계값': 4,
        'F1': 4,
        'PR-AUC': 4,
        '이벤트 예측 비율': 4,
        '기존 평균 별점': 4,
        '정제 후 평균 별점': 4,
        '평균 별점 변화량': 4,
        '평균 절대 변화량': 4,
        '최대 절대 변화량': 4,
    })
)

print('메인 비교: 제안 모델 vs validation 기준 선택 모델')
display(main_comparison_display)


메인 비교: 제안 모델 vs validation 기준 선택 모델


,모델,모델 구분,임계값,F1,PR-AUC,이벤트 예측 수,이벤트 예측 비율,기존 평균 별점,정제 후 평균 별점,평균 별점 변화량,평균 절대 변화량,최대 절대 변화량
0,제안 모델 (proposed_hybrid_mlp_04: kcbert_pca+meta...,proposed,0.5009,0.4400,0.4029,586,0.4416,4.896,4.8711,-0.0078,0.0211,0.4166
1,선택 모델 (baseline_2_metadata_only_mlp: metadata_...,selected,0.5011,0.5014,0.3564,983,0.7408,4.896,4.8793,0.0003,0.0042,0.0845


## 11. 제안/선택 모델 후보 2 결과 출력 및 저장

- 제안 모델과 선택 모델에 후보 2를 적용한 결과를 출력하고 CSV로 저장한다.
- 이 표에는 F1, PR-AUC와 별점 정제 변화량이 함께 들어간다.
- 저장 파일: `outputs/rating_refinement_all_model_summary.csv`


In [11]:
appendix_comparison_display = (
    refinement_summary[[
        'model_label',
        'model_role',
        'threshold',
        'review_count',
        'f1',
        'pr_auc',
        'event_pred_count',
        'event_pred_rate',
        'store_count',
        'original_mean_rating',
        'refined_mean_rating',
        'mean_rating_delta',
        'mean_abs_rating_delta',
        'max_abs_rating_delta',
        'fallback_store_count',
    ]]
    .sort_values(['model_role', 'model_label'])
    .reset_index(drop=True)
    .rename(columns={
        'model_label': '모델',
        'model_role': '모델 구분',
        'threshold': '임계값',
        'review_count': '리뷰 수',
        'f1': 'F1',
        'pr_auc': 'PR-AUC',
        'event_pred_count': '이벤트 예측 수',
        'event_pred_rate': '이벤트 예측 비율',
        'store_count': '가게 수',
        'original_mean_rating': '기존 평균 별점',
        'refined_mean_rating': '정제 후 평균 별점',
        'mean_rating_delta': '평균 별점 변화량',
        'mean_abs_rating_delta': '평균 절대 변화량',
        'max_abs_rating_delta': '최대 절대 변화량',
        'fallback_store_count': 'fallback 가게 수',
    })
    .round({
        '임계값': 4,
        'F1': 4,
        'PR-AUC': 4,
        '이벤트 예측 비율': 4,
        '기존 평균 별점': 4,
        '정제 후 평균 별점': 4,
        '평균 별점 변화량': 4,
        '평균 절대 변화량': 4,
        '최대 절대 변화량': 4,
    })
)

all_model_summary_path = 'outputs/rating_refinement_all_model_summary.csv'
appendix_comparison_display.to_csv(all_model_summary_path, index=False, encoding='utf-8-sig')
print(f'저장 완료: {all_model_summary_path} ({len(appendix_comparison_display)}행)')
display(appendix_comparison_display)


저장 완료: outputs/rating_refinement_all_model_summary.csv (2행)


,모델,모델 구분,임계값,리뷰 수,F1,PR-AUC,이벤트 예측 수,이벤트 예측 비율,가게 수,기존 평균 별점,정제 후 평균 별점,평균 별점 변화량,평균 절대 변화량,최대 절대 변화량,fallback 가게 수
0,제안 모델 (proposed_hybrid_mlp_04: kcbert_pca+meta...,proposed,0.5009,1327,0.4400,0.4029,586,0.4416,118,4.896,4.8711,-0.0078,0.0211,0.4166,0
1,선택 모델 (baseline_2_metadata_only_mlp: metadata_...,selected,0.5011,1327,0.5014,0.3564,983,0.7408,118,4.896,4.8793,0.0003,0.0042,0.0845,0


- `리뷰 수`: test 데이터에서 해당 모델이 평가한 리뷰 개수이다.
- `가게 수`: test 데이터에 포함된 고유 가게 개수이다.
- `임계값`: 이벤트 리뷰로 판단하는 확률 기준이다.
- `F1`: 이벤트 리뷰 탐지에서 precision과 recall의 균형을 보는 지표이다.
- `PR-AUC`: 이벤트 리뷰 탐지 성능을 확률 순위 관점에서 보는 보조 지표이다.
- `이벤트 예측 수`: 모델이 이벤트 리뷰라고 판단한 리뷰 개수이다.
- `이벤트 예측 비율`: 전체 test 리뷰 중 이벤트 리뷰라고 판단한 비율이다.
- `기존 평균 별점`: 후보 2 적용 전, test 데이터의 가게별 평균 별점들을 평균낸 값이다.
- `정제 후 평균 별점`: 후보 2 적용 후, 가게별 정제 별점들을 평균낸 값이다.
- `평균 별점 변화량`: 가게별로 `정제 후 별점 - 기존 별점`을 계산한 뒤 평균낸 값이다.
- `평균 절대 변화량`: 별점이 오른/내린 방향은 무시하고, 가게별 별점이 평균적으로 얼마나 움직였는지 나타낸 값이다.
- `최대 절대 변화량`: 후보 2 적용 후 별점이 가장 크게 바뀐 가게의 변화 크기이다.
- `fallback 가게 수`: 후보 2 가중치 합이 0이라 기존 평균 별점을 그대로 사용한 가게 수이다.

## 12. 오답 리뷰 샘플 출력 및 저장

- 제안 모델과 선택 모델의 FP/FN 리뷰를 출력하고 CSV로 저장한다.
- FP는 일반 리뷰인데 이벤트 리뷰로 예측한 경우이다.
- FN은 이벤트 리뷰인데 일반 리뷰로 놓친 경우이다.
- 저장 파일: `outputs/rating_refinement_error_samples.csv`


In [12]:
error_samples = (
    review_refinement_detail[
        review_refinement_detail['model_role'].isin(['proposed', 'selected'])
        & review_refinement_detail['error_type'].isin(['FP', 'FN'])
    ]
    .sort_values(['model_role', 'error_type', 'event_prob'], ascending=[True, True, False])
    .groupby(['model_label', 'error_type'], group_keys=False)
    .head(10)
)

error_sample_cols = [
    'model_label',
    'error_type',
    'raw_index',
    'store_name',
    'rating',
    'review_text',
    'cleaned_review_text',
    'actual_label_name',
    'event_pred_name',
    'event_prob',
    'candidate_2_weight',
    'original_store_rating',
    'candidate_2_store_refined_rating',
]

error_samples_output = (
    error_samples[error_sample_cols]
    .rename(columns={
        'model_label': '모델',
        'error_type': '오답 유형',
        'raw_index': '원본 행 번호',
        'store_name': '가게명',
        'rating': '리뷰 별점',
        'review_text': '리뷰 원문',
        'cleaned_review_text': '정제 리뷰',
        'actual_label_name': '실제 라벨',
        'event_pred_name': '예측 라벨',
        'event_prob': '이벤트성 확률',
        'candidate_2_weight': '후보 2 가중치',
        'original_store_rating': '기존 평균 별점',
        'candidate_2_store_refined_rating': '후보 2 정제 후 별점',
    })
    .round({
        '이벤트성 확률': 4,
        '후보 2 가중치': 4,
        '기존 평균 별점': 4,
        '후보 2 정제 후 별점': 4,
    })
)

error_samples_path = 'outputs/rating_refinement_error_samples.csv'
error_samples_output.to_csv(error_samples_path, index=False, encoding='utf-8-sig')
print(f'저장 완료: {error_samples_path} ({len(error_samples_output)}행)')
display(error_samples_output)


저장 완료: outputs/rating_refinement_error_samples.csv (40행)


,모델,오답 유형,원본 행 번호,가게명,리뷰 별점,리뷰 원문,정제 리뷰,실제 라벨,예측 라벨,이벤트성 확률,후보 2 가중치,기존 평균 별점,후보 2 정제 후 별점
788,제안 모델 (proposed_hybrid_mlp_04: kcbert_pca+meta...,FN,1543,https://s.baemin.com/Lb000iNrcrSW0,5.0,맛있어요👍👍👍,맛있어요,이벤트 리뷰(1),일반 예측(0),0.4997,0.5003,5.0000,5.0000
380,제안 모델 (proposed_hybrid_mlp_04: kcbert_pca+meta...,FN,5178,난곡반점,5.0,사천탕수육 맛집을 찾았습니다!!\n글구 리뷰로 보내주신 미니짜장밥\n퀄리티 대박입니...,사천탕수육 맛집을 찾았습니다!! 글구 리뷰로 보내주신 미니짜장밥 퀄리티 대박입니다~...,이벤트 리뷰(1),일반 예측(0),0.4992,0.5008,5.0000,5.0000
1112,제안 모델 (proposed_hybrid_mlp_04: kcbert_pca+meta...,FN,4814,이프유캔 Coffee&Dessert 강남본점,5.0,근처에서 촬영하다가 리뷰가 좋길래 시켜봤는데\n리뷰가 좋은건 다 이유가 있는 것 같...,근처에서 촬영하다가 리뷰가 좋길래 시켜봤는데 리뷰가 좋은건 다 이유가 있는 것 같네...,이벤트 리뷰(1),일반 예측(0),0.4989,0.5011,4.9231,4.9536
313,제안 모델 (proposed_hybrid_mlp_04: kcbert_pca+meta...,FN,2108,https://s.baemin.com/sO000gAx6rl9w,5.0,오자마자 먹어버려서 사진이 없네용...ㅠㅠ\n뿌링핫도그 넘 맛있고 떡도 완전 쫄깃해...,오자마자 먹어버려서 사진이 없네용...ㅠㅠ 뿌링핫도그 넘 맛있고 떡도 완전 쫄깃해요...,이벤트 리뷰(1),일반 예측(0),0.4989,0.5011,5.0000,5.0000
1240,제안 모델 (proposed_hybrid_mlp_04: kcbert_pca+meta...,FN,2026,https://s.baemin.com/zY000CmvG66LK,5.0,달다구리 맛있네용☺️,달다구리 맛있네용,이벤트 리뷰(1),일반 예측(0),0.4975,0.5025,4.6000,4.7359
1257,제안 모델 (proposed_hybrid_mlp_04: kcbert_pca+meta...,FN,2749,https://s.baemin.com/Ha000fnAkEtl5,5.0,든든하고 맛있어요! 후레이크?좀더 많으면좋을것같네요,든든하고 맛있어요! 후레이크?좀더 많으면좋을것같네요,이벤트 리뷰(1),일반 예측(0),0.4969,0.5031,4.8000,4.7622
337,제안 모델 (proposed_hybrid_mlp_04: kcbert_pca+meta...,FN,3370,광어군 오백양,5.0,와 첫 주문이였는데 왜 여기를 이제 알게된건지 한탄스럽네요.\n양도 많고 회도 두껍...,와 첫 주문이였는데 왜 여기를 이제 알게된건지 한탄스럽네요. 양도 많고 회도 두껍고...,이벤트 리뷰(1),일반 예측(0),0.4958,0.5042,5.0000,5.0000
613,제안 모델 (proposed_hybrid_mlp_04: kcbert_pca+meta...,FN,6888,인쌩맥주 수유역점,5.0,다맛있고 좋았어요~~!!,다맛있고 좋았어요~~!!,이벤트 리뷰(1),일반 예측(0),0.4938,0.5062,5.0000,5.0000
32,제안 모델 (proposed_hybrid_mlp_04: kcbert_pca+meta...,FN,5439,드디어 찾은 인생 최고의 중국집,5.0,헐 짜장면 맛집 발견!! 탕수육도 탕수육인데 짜장이 진짜 맛있어요 요즘 그 면이랑 ...,헐 짜장면 맛집 발견!! 탕수육도 탕수육인데 짜장이 진짜 맛있어요 요즘 그 면이랑 ...,이벤트 리뷰(1),일반 예측(0),0.4901,0.5099,4.9333,4.9453
777,제안 모델 (proposed_hybrid_mlp_04: kcbert_pca+meta...,FN,1659,https://s.baemin.com/48000frmedErp,5.0,색다른 두쫀쿠 맛이 첨이라 시켜봤는데 너무 맛있어요!!! 고소짭짤 또 시킬거같아요 ㅎㅎ,색다른 두쫀쿠 맛이 첨이라 시켜봤는데 너무 맛있어요!!! 고소짭짤 또 시킬거같아요 ㅎㅎ,이벤트 리뷰(1),일반 예측(0),0.4900,0.5100,4.9167,4.8663


## 13. 별점 변화가 큰 가게 샘플 출력 및 저장

- 제안 모델과 선택 모델에서 후보 2 적용 후 별점 변화량이 큰 가게 샘플을 출력하고 CSV로 저장한다.
- 변화량이 큰 가게는 이벤트 확률 기반 가중치가 평균 별점에 크게 영향을 준 사례이다.
- 저장 파일: `outputs/rating_refinement_top_changed_stores.csv`


In [13]:
top_changed_samples = (
    all_store_results[all_store_results['model_role'].isin(['proposed', 'selected'])]
    .assign(abs_rating_delta=lambda df: df['rating_delta'].abs())
    .sort_values('abs_rating_delta', ascending=False)
    [[
        'model_label',
        'candidate_label',
        'store_id',
        'store_name',
        'review_count',
        'original_mean_rating',
        'refined_rating',
        'rating_delta',
        'fallback_used',
    ]]
    .head(20)
    .rename(columns={
        'model_label': '모델',
        'candidate_label': '정제 방식',
        'store_id': '가게 ID',
        'store_name': '가게명',
        'review_count': '리뷰 수',
        'original_mean_rating': '기존 평균 별점',
        'refined_rating': '후보 2 정제 후 별점',
        'rating_delta': '별점 변화량',
        'fallback_used': 'fallback 사용 여부',
    })
    .round({
        '기존 평균 별점': 4,
        '후보 2 정제 후 별점': 4,
        '별점 변화량': 4,
    })
)

top_changed_samples_path = 'outputs/rating_refinement_top_changed_stores.csv'
top_changed_samples.to_csv(top_changed_samples_path, index=False, encoding='utf-8-sig')
print(f'저장 완료: {top_changed_samples_path} ({len(top_changed_samples)}행)')
display(top_changed_samples)


저장 완료: outputs/rating_refinement_top_changed_stores.csv (20행)


,모델,정제 방식,가게 ID,가게명,리뷰 수,기존 평균 별점,후보 2 정제 후 별점,별점 변화량,fallback 사용 여부
36,제안 모델 (proposed_hybrid_mlp_04: kcbert_pca+meta...,후보 2: 이벤트 확률 기반 가중 평균,https://s.baemin.com/pG000flG524ai,https://s.baemin.com/pG000flG524ai,4,3.7500,3.3334,-0.4166,False
31,제안 모델 (proposed_hybrid_mlp_04: kcbert_pca+meta...,후보 2: 이벤트 확률 기반 가중 평균,https://s.baemin.com/fk000fkrS8JGB,https://s.baemin.com/fk000fkrS8JGB,4,2.7500,3.0021,0.2521,False
18,제안 모델 (proposed_hybrid_mlp_04: kcbert_pca+meta...,후보 2: 이벤트 확률 기반 가중 평균,https://s.baemin.com/Q6000jyZx2LMo,https://s.baemin.com/Q6000jyZx2LMo,10,4.5000,4.2552,-0.2448,False
45,제안 모델 (proposed_hybrid_mlp_04: kcbert_pca+meta...,후보 2: 이벤트 확률 기반 가중 평균,https://s.baemin.com/xp000fAWYLfLe,https://s.baemin.com/xp000fAWYLfLe,13,3.6923,3.4782,-0.2141,False
33,제안 모델 (proposed_hybrid_mlp_04: kcbert_pca+meta...,후보 2: 이벤트 확률 기반 가중 평균,https://s.baemin.com/iE000iHSmeDVY,https://s.baemin.com/iE000iHSmeDVY,7,4.4286,4.2689,-0.1597,False
46,제안 모델 (proposed_hybrid_mlp_04: kcbert_pca+meta...,후보 2: 이벤트 확률 기반 가중 평균,https://s.baemin.com/zY000CmvG66LK,https://s.baemin.com/zY000CmvG66LK,10,4.6000,4.7359,0.1359,False
24,제안 모델 (proposed_hybrid_mlp_04: kcbert_pca+meta...,후보 2: 이벤트 확률 기반 가중 평균,https://s.baemin.com/Vp000ty8vKgb7,https://s.baemin.com/Vp000ty8vKgb7,9,4.1111,4.2417,0.1306,False
73,제안 모델 (proposed_hybrid_mlp_04: kcbert_pca+meta...,후보 2: 이벤트 확률 기반 가중 평균,빽보이피자 봉천점,빽보이피자 봉천점,13,4.8462,4.7162,-0.1300,False
154,선택 모델 (baseline_2_metadata_only_mlp: metadata_...,후보 2: 이벤트 확률 기반 가중 평균,https://s.baemin.com/pG000flG524ai,https://s.baemin.com/pG000flG524ai,4,3.7500,3.8345,0.0845,False
44,제안 모델 (proposed_hybrid_mlp_04: kcbert_pca+meta...,후보 2: 이벤트 확률 기반 가중 평균,https://s.baemin.com/wp000eZswFf3l,https://s.baemin.com/wp000eZswFf3l,14,4.7143,4.6334,-0.0809,False
